In [53]:
!pip3 install flask

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 102 kB 1.2 MB/s eta 0:00:01
You should consider upgrading via the '/Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip' command.


In [22]:
import pandas as pd
import requests
import json
import time
import Levenshtein
from urllib.parse import urlparse
from flask import Flask, jsonify
from requests.auth import HTTPBasicAuth

In [60]:
# Function to fetch exchanges with pagination
def fetch_exchanges(url):
    page = 1
    all_exchanges = []
    
    while True:
        # Add the page parameter to the request
        params = {"per_page": 250, "page": page} 
        try:
            # Make the request
            response = requests.get(url, params=params)
            # Check if the request was successful
            if response.status_code == 200:
                data = response.json()
                # If no data is returned, stop the loop
                if not data:
                    break
                # Add the fetched data to the list
                all_exchanges.extend(data)
                print(f"Fetched page {page} with {len(data)} exchanges.")
                # Move to the next page
                page += 1
                # Add a delay to avoid hitting the rate limit
                time.sleep(1) 
    
            elif response.status_code == 429:
                # Handle rate limit error
                print("Rate limit exceeded. Waiting for 10 seconds before retrying...")
                time.sleep(60)  # Wait 60 seconds before retrying
            
            else:
                # Handle other errors
                print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
                break
        
        except requests.exceptions.RequestException as e:
            # Handle network errors
            print(f"Network error: {e}")
            break
    
    return all_exchanges

# Function to fetch exchanges from DeFi Llama
def fetch_exchanges_II(url):
    # Make the request
    response = requests.get(url)
    if response.status_code == 200:
        # Return the JSON data
        return response.json() 
    else:
        print(f"Failed to fetch data. Status code: {response.status_code}. Reason: {response.reason}")
        
        return []

# Function to fetch the blacklist url
def blacklist_toMap(items):
    # Create a list to store the name-url mappings
    blacklist_map = []

    # Process each URL in the blacklist
    for url in items:
        # Extract the text before the domain extension
        name = url.split('.')[0]
        # Create a dictionary for the name and URL
        blacklist_map.append({'name': name, 'url': url})

    return blacklist_map


def fetch_bank_institutions(auth_url, institutions_url, client_id, client_secret):
    # authenticate and get access token
    auth_response = requests.post(auth_url, auth=HTTPBasicAuth(client_id, client_secret), data={"grant_type": "client_credentials"})
    
    if auth_response.status_code != 200:
        return f"Authentication failed: {auth_response.text}"
    
    access_token = auth_response.json().get("access_token")
    headers = {"Authorization": f"Bearer {access_token}"}
    
    # fetch financial institutions
    response = requests.get(institutions_url, headers=headers)
    
    if response.status_code != 200:
        return f"Failed to fetch institutions: {response.text}"
    
    institutions = response.json()
    return [{"id": inst["id"], "name": inst["name"], "countries": inst["countries"]} for inst in institutions]




# function to extract domain
extract_domain = lambda url: urlparse(url if "://" in url else f"https://{url}").netloc.replace("www.", "")

# Define the isTyposquatting function
def isTyposquatting(legit_domains, scam_domain, threshold=2):
    for legit_domain in legit_domains:
        if Levenshtein.ratio(legit_domain, scam_domain) <= threshold:
            return True
    return False

def format_domain(domain):
    if not domain.startswith(('https://', 'http://', 'www.')):
        return 'https://' + domain
    return domain


In [ ]:
auth_url = "https://api.yapily.com/auth/token"

institutions_url = "https://api.yapily.com/institutions"

fetch_bank_institutions()


In [24]:
# fetch centralized exchanges from coingecko
cex_url = "https://api.coingecko.com/api/v3/exchanges"
gecko_data = fetch_exchanges(cex_url)
# view the first item of the data
gecko_data[0]

Fetched page 1 with 250 exchanges.
Fetched page 2 with 250 exchanges.
Fetched page 3 with 250 exchanges.
Fetched page 4 with 215 exchanges.


{'id': 'binance',
 'name': 'Binance',
 'year_established': 2017,
 'country': 'Cayman Islands',
 'description': 'One of the world’s largest cryptocurrency exchanges by trading volume, offering a wide range of services including spot, futures, and staking options.',
 'url': 'https://www.binance.com/',
 'image': 'https://coin-images.coingecko.com/markets/images/52/small/binance.jpg?1706864274',
 'has_trading_incentive': False,
 'trust_score': 10,
 'trust_score_rank': 1,
 'trade_volume_24h_btc': 277145.4235946333,
 'trade_volume_24h_btc_normalized': 185668.88791891848}

In [25]:
# fetch every exchange protocol from defi llama
url = "https://api.llama.fi/protocols"
llama_data = fetch_exchanges_II(url)
# view the first item of the data
llama_data[0]

{'id': '2269',
 'name': 'Binance CEX',
 'address': None,
 'symbol': '-',
 'url': 'https://www.binance.com',
 'description': 'Binance is a cryptocurrency exchange which is the largest exchange in the world in terms of daily trading volume of cryptocurrencies',
 'chain': 'Multi-Chain',
 'logo': 'https://icons.llama.fi/binance-cex.jpg',
 'audits': '0',
 'audit_note': None,
 'gecko_id': None,
 'cmcId': None,
 'category': 'CEX',
 'chains': ['Bitcoin',
  'Ethereum',
  'Binance',
  'Solana',
  'Ripple',
  'Tron',
  'Doge',
  'Base',
  'Arbitrum',
  'Optimism',
  'Avalanche',
  'Litecoin',
  'Polkadot',
  'Near',
  'Polygon',
  'Aptos',
  'Algorand',
  'Starknet',
  'Manta',
  'Op_Bnb',
  'zkSync Era',
  'Fantom'],
 'module': 'binance/index.js',
 'twitter': 'binance',
 'forkedFrom': [],
 'oracles': [],
 'listedAt': 1668170565,
 'methodology': 'We collect the wallets from this binance blog post https://www.binance.com/en/blog/community/our-commitment-to-transparency-2895840147147652626. We are 

In [26]:
crypto_exchanges = llama_data + gecko_data

# Print the total number of exchanges fetched
print(f"Total exchanges fetched: {len(crypto_exchanges)}")

Total exchanges fetched: 6571


In [27]:
# Create a set to store unique URLs
unique_urls = set()

# Add URLs from cex_data to the set
for exchange in gecko_data:
    # Normalize by stripping trailing slashes
    unique_urls.add(exchange['url'].rstrip('/')) 

# Create a list to store unique exchanges
unique_exchanges = []

# Add exchanges from cex_data to the unique list
for exchange in gecko_data:
    if exchange['url'].rstrip('/') in unique_urls:
        unique_exchanges.append(exchange)

# Add exchanges from ex_data to the unique list
for exchange in llama_data:
    if exchange['url'].rstrip('/') in unique_urls:
        unique_exchanges.append(exchange)

# Print the total number of unique exchanges fetched
print(f"Total unique exchanges fetched: {len(unique_exchanges)}")

Total unique exchanges fetched: 1191


In [28]:
with open("legit-exchanges.json", "w") as file:
    json.dump(unique_exchanges, file)

In [64]:

df_raw = pd.read_json("legit-exchanges.json")

# copy the name and urls columns to a new dataframe
legit_df = df_raw[['name','url']].copy()

# apply the lambda function to the 'url' column to extract the domain
# legit_df["domain"] = legit_df["url"].apply(extract_domain)

legit_df["url"] = legit_df["url"].apply(format_domain)

# assign string labels to legitimate urls
legit_df.loc[:, 'label'] = 'legit'

# assign string labels to legitimate urls
legit_df.loc[:, 'label_no'] = '0'

legit_df["url"] 

0            https://www.binance.com/
1           https://www.coinbase.com/
2                 https://www.okx.com
3             https://www.bitget.com/
4               https://www.huobi.com
                    ...              
1186         https://app.dem.exchange
1187       https://vertexprotocol.com
1188    https://lfj.gg/arbitrum/trade
1189           https://sologenic.org/
1190      https://app.pulsex.com/swap
Name: url, Length: 1191, dtype: object

In [30]:
# url to fetch scam urls from eth-phishing-detect
scam_url = "https://raw.githubusercontent.com/MetaMask/eth-phishing-detect/master/src/config.json"
scam_ex = fetch_exchanges_II(scam_url)

# fetch the list of blacklist urls
blacklist = scam_ex["blacklist"]

In [31]:
blacklist = blacklist_toMap(blacklist)

In [32]:
with open("scam-exchanges.json", "w") as file:
    json.dump(blacklist, file)

In [62]:
blacklist_raw = pd.read_json("scam-exchanges.json")

# copy the name and urls columns to a new dataframe
scam_df = blacklist_raw[['name','url']].copy()

scam_df["url"] = scam_df["url"].apply(format_domain)

# apply the lambda function to the 'url' column to extract the domain
# scam_df["domain"] = scam_df["url"].apply(extract_domain)

# check if a scam domain contains typosquatting 
# scam_df["isTyposquatting"] = scam_df["domain"].apply(lambda domain : isTyposquatting(legit_df["domain"].tolist(), domain))

# assign string labels to legitimate urls
scam_df.loc[:, 'label'] = 'scam'

# assign string labels to legitimate urls
scam_df.loc[:, 'label_no'] = '1'


0             https://ogntoken-migration.icu
1         https://fazla-rabby-rady.github.io
2                    https://polymaraket.com
3                     https://predictdex.com
4                      https://polymarket.mx
                         ...                
198091                 https://taikoh-ss.com
198092           https://taiko.mintstory.xyz
198093              https://virgocx-page.com
198094              https://tronsniper.cloud
198095            https://walletguard.com.au
Name: url, Length: 198096, dtype: object

In [34]:
label_map = {0: "legit", 1: "scam"}
label_map

{0: 'legit', 1: 'scam'}

In [50]:
# merge both the legit and scam urls together
urls_df = pd.concat([legit_df, scam_df], ignore_index=True)

# shuffle the urls across the dataframe
urls_df = urls_df.sample(frac=1, random_state=42).reset_index(drop=True)

urls_df

,name,url,label,label_no
0,doge20,doge20.pages.dev,scam,1
1,snap-dioneprotocol,snap-dioneprotocol.pages.dev,scam,1
2,bitcminetirix,bitcminetirix.online,scam,1
3,claim-xrp,claim-xrp.xyz,scam,1
4,madpack,madpack.site,scam,1
...,...,...,...,...
199282,solana-ecosystem,solana-ecosystem.pages.dev,scam,1
199283,app-beoble,app-beoble.com,scam,1
199284,revshares-dashboard,revshares-dashboard.pages.dev,scam,1
199285,taobridge,taobridge.org,scam,1


In [56]:
urls_df.to_json('crypto_data.json', orient='records', lines=False)